# Ko-miracl RAG — 생성(Generation) 파이프라인

리포의 `rag/` 모듈과 `prompts.py`를 **그대로 가져다 쓰는** 얇은 실행 노트북입니다.
Colab 불필요 — **로컬 CPU**에서 돕니다. 생성 LLM은 로컬 로드 없이 OpenAI API를 호출합니다.

```
사용자의 추상적 질문
   └─▶ ① LLM 질의 재작성 (rewrite)          rag.generation.rewrite_query
        └─▶ ② retriever 검색 (ChromaDB)      rag.index.retrieve
             └─▶ ③ 프롬프트 생성 (문서 주입)  rag.generation.SYSTEM_PROMPTS (strict)
                  └─▶ ④ LLM 응답 생성         rag.llm.chat
                       └─▶ ⑤ RAGAS 평가        LLM-as-judge (gpt-5.4-mini)
```

- **임베딩(검색)**: `BAAI/bge-m3` — 평가 서브셋만 색인 (CPU에서 수 분)
- **LLM(생성)**: OpenAI `gpt-4o-mini` API, temperature=0 — `.env`의 `OPENAI_API_KEY` 필요
- 사전 준비: `pip install -r requirements.txt` (datasets, chromadb, sentence-transformers, ragas 등)

## 0. 설정 — 리포 모듈 로드

In [ ]:
import json, os, random
import numpy as np, pandas as pd
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), ".env 에 OPENAI_API_KEY 를 채워주세요"

from rag.config import DATA, SEED, GenerationConfig

random.seed(SEED); np.random.seed(SEED)
cfg = GenerationConfig()
DEVICE = "cpu"               # 로컬 CPU 실행 (GPU 있으면 "cuda")

N_EVAL = cfg.n_eval_queries  # 평가 쿼리 수 (30)
NEG_POOL = 500               # CPU 임베딩이 병목이라 축소. GPU면 cfg.neg_pool_size(3000)
BATCH = 32                   # CPU 임베딩 배치
print("device:", DEVICE, "| embed:", cfg.embed_model)

### 데이터 로드 — HF에서 정답셋(queries/qrels) + 코퍼스 서브셋(정답 문서+네거티브 풀)만 스트리밍 수집

In [ ]:
from rag.data import (build_gold, collect_corpus, load_qrels,
                      load_queries, needed_corpus_ids, sample_pos_queries)

queries = load_queries(DATA)
dev_qrels = load_qrels(DATA, DATA.dev_split)
eval_qids = sample_pos_queries(dev_qrels, DATA, N_EVAL, SEED)
gold = build_gold(dev_qrels, DATA, eval_qids)
needed = needed_corpus_ids(gold)

corpus_text, corpus_title = collect_corpus(DATA, needed, NEG_POOL)
index_cids = list(corpus_text)
found = sum(c in corpus_text for c in needed)
print(f"색인 대상 {len(index_cids)}개 청크 (정답 {found}/{len(needed)} + 네거티브 풀 {NEG_POOL})")

### 임베딩 · ChromaDB · retriever (재사용) — 최초 1회 bge-m3 다운로드(~2GB), CPU 색인 수 분

In [ ]:
import chromadb
from rag.embedding import load_embedder
from rag.index import build_collection, retrieve

model = load_embedder(cfg.embed_model, DEVICE, cfg.max_seq_len)
client = chromadb.Client()
collection = build_collection(
    client, "komiracl-gen-cpu", model, index_cids, corpus_text, corpus_title,
    passage_prefix=cfg.passage_prefix, batch_size=BATCH,
)
print("색인 완료:", collection.count(), "청크")

## 1. 생성 LLM — OpenAI API (`rag.llm.chat`)
로컬 모델 로드가 없습니다. `gpt-4o-mini`를 API로 호출합니다 (temperature=0, 429/5xx 자동 재시도).

In [ ]:
from rag.llm import LLM_MODEL, chat
print("LLM:", LLM_MODEL)

## 2. ① 질의 재작성 (`rag.generation.rewrite_query`)

In [ ]:
from rag.generation import rag_answer, rewrite_query
print(rewrite_query("물 몇도에 끓음?"))   # API 연결 스모크 겸 데모

## 3. ③ 프롬프트
고정 필수 규칙: ① 참고 문서만을 기반으로 답변 ② 문서에 없으면 '제공된 문서에서 찾을 수 없습니다'라고만 답변.
`rag_answer`는 이 규칙의 strict 프롬프트(`rag.generation.SYSTEM_PROMPTS`)를 사용합니다.

In [ ]:
from rag.generation import SYSTEM_PROMPTS
print(SYSTEM_PROMPTS["strict"])

## 4. 전체 파이프라인 (`rag.generation.rag_answer`: 재작성 → 검색 → strict 프롬프트 → 생성)

In [ ]:
def show(question, k=cfg.top_k):
    r = rag_answer(question, collection, model, cfg.query_prefix, k=k)
    print("① 원 질문     :", r["question"])
    print("① 재작성 질의 :", r["rewritten"])
    print("② 검색 결과 (score = 1 - distance):")
    for i, c in enumerate(r["contexts"], 1):
        print(f"   [{i}] score={c['score']:.4f} id={c['corpus_id']} | {c['title']}")
    print("\n④ 생성 응답 :\n" + r["answer"])
    return r

## 5. 실행 예시

In [ ]:
_ = show(queries[eval_qids[12]])   # 예시 A: dev 실제 질문 (정답 문서가 색인에 포함)

In [ ]:
_ = show("물 몇도에 끓음?")          # 예시 B: 추상적/구어 질문

In [ ]:
_ = show("키보드 주문제작 하고 싶어")

## 6. 평가 — RAGAS (LLM-as-judge) 스모크 테스트
소량 샘플로 **judge 평가 파이프라인**이 도는지 확인합니다. judge는 `evaluation/run_ragas_eval.py`와
**같은 상수를 import**해 고정합니다(`gpt-5.4-mini`) — 노트북과 본 평가가 같은 기준으로 채점됩니다.

| 지표 | 의미 | 방향 |
|---|---|---|
| **faithfulness (충실도)** | 답변의 주장들이 검색 문서에 근거하는 비율 | 높을수록 좋음 |
| **answer relevancy** | 답변이 질문 의도와 관련된 정도 | 높을수록 좋음 |

비용 주의: judge 호출이 생성보다 비쌉니다. 여기선 소량 스모크만 —
본 평가는 `evaluation/generate_answers.py`(대량 생성) → `evaluation/run_ragas_eval.py`(샘플링, 200건 상한)로.

In [ ]:
# ① 평가 입력 생성 — rag_answer 로 소량(N_JUDGE)만 (gpt-4o-mini, 비용 몇 센트)
from tqdm.auto import tqdm

N_JUDGE = 5   # 스모크용. 늘리면 생성·judge 비용이 비례해서 증가
records = []
for qid in tqdm(eval_qids[:N_JUDGE], desc="생성"):
    r = rag_answer(queries[qid], collection, model, cfg.query_prefix, k=cfg.top_k)
    records.append({"qid": qid, "question": r["question"], "answer": r["answer"],
                    "contexts": [c["text"] for c in r["contexts"]]})

# run_ragas_eval.py 입력 형식과 동일하게 저장 → 스크립트로도 그대로 평가 가능
os.makedirs("result", exist_ok=True)
with open("result/generations_notebook.jsonl", "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("예시 답변:", records[0]["answer"])

In [ ]:
# ② RAGAS 평가 — judge 모델은 본 평가 스크립트와 동일 상수로 고정
from evaluation.run_ragas_eval import JUDGE_EMBED, JUDGE_MODEL
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, ResponseRelevancy

judge = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model=JUDGE_EMBED))

ds = EvaluationDataset.from_list([
    {"user_input": r["question"], "response": r["answer"],
     "retrieved_contexts": r["contexts"]}
    for r in records if r["answer"] and r["contexts"]   # 빈 답/무근거는 채점 불가
])
result = evaluate(ds, metrics=[Faithfulness(llm=judge),
                               ResponseRelevancy(llm=judge, embeddings=emb)])
df = result.to_pandas()
df

In [ ]:
# ③ 건별 저장 + 평균 — 기권('찾을 수 없습니다') 답변이 많으면 평균 해석에 주의
df.to_csv("result/ragas_notebook_smoke.csv", index=False, encoding="utf-8-sig")
print("저장: result/ragas_notebook_smoke.csv")
df.select_dtypes("number").mean().round(3)

### 더 해볼 것
- 재작성을 **다중 질의(multi-query)** 로 확장해 검색 recall 향상
- 검색 API(dense→리랭크→윈도우, `rag_engine.run_pipeline`) 경로와 결과 비교
- 충실도(faithfulness) 낮은 답변 원문 분석 → 시스템 프롬프트 보완
- 대량 생성(`evaluation/generate_answers.py`) → 본 RAGAS 평가(`evaluation/run_ragas_eval.py`)